# Feedback & monitoring — closing the production loop

*Part of the `gen_ai/` track (the finale). Prerequisites: `a_tracing_quickstart`, `c_genai_evaluation`.*

## The problem: a test set isn't production

`c_genai_evaluation` scored a **fixed** set of questions before release. But once real users hit the app, two things happen that no test set captures:

- **Users react** — they thumbs-up/down answers, edit them, file complaints. That signal is gold, and it should attach to the exact trace it's about.
- **Quality drifts** — a model update, a prompt change, or weird real inputs quietly degrade answers. You only catch it if you **keep scoring live traffic**.

Both attach to **traces** (from `a_`/`b_`/`d_`) as **assessments**: a name, a value, a source (a *human*, an *LLM judge*, or *code*), and a rationale. This notebook wires the loop: collect feedback on traces, then monitor live traffic by re-running scorers over recent traces.

| Term | One-line meaning |
|---|---|
| **assessment / feedback** | A judgment attached to a trace — `log_feedback(trace_id, name, value, source, rationale)`. |
| **source** | Who judged: `HUMAN`, `LLM_JUDGE`, or `CODE`. |
| **monitoring** | Re-running scorers over recent production **traces** to track quality over time. |

## Prerequisites

- **Server on 5001**, **Ollama** (`qwen3:8b`) to produce traces, and a **capable judge** (Azure, as in `c_`) for monitoring. Credentials from `.env`. No new dependencies.

In [ ]:
import os, re
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(), override=True)
os.environ["AZURE_API_KEY"]     = os.environ["AZURE_OPENAI_API_KEY"]
os.environ["AZURE_API_BASE"]    = os.environ["AZURE_OPENAI_BASE_URL"].removesuffix("/openai").rstrip("/")
os.environ["AZURE_API_VERSION"] = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21")
JUDGE = f"azure:/{os.environ.get('AZURE_OPENAI_LIGHT_MODEL', 'gpt-5.4-nano')}"

import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("genai-feedback-monitoring")
mlflow.openai.autolog()

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama", timeout=60, max_retries=0)
MODEL = "qwen3:8b"

## Simulate some production traffic

Run a traced app a few times — these stand in for real user requests. `flush_trace_async_logging()` makes sure the traces are persisted before we attach anything to them (traces export asynchronously).

In [ ]:
@mlflow.trace
def app(question: str) -> str:
    r = client.chat.completions.create(
        model=MODEL, max_tokens=256,
        messages=[{"role": "user", "content": f"{question} /no_think"}])
    return re.sub(r"<think>.*?</think>", "", r.choices[0].message.content, flags=re.DOTALL).strip()

for q in ["What is MLflow Tracing?",
          "What is an MLflow model registry?",
          "What does mlflow models serve do?"]:
    app(q)

mlflow.flush_trace_async_logging()                 # persist before attaching feedback
traces = mlflow.search_traces(max_results=10)      # recent traffic in this experiment
print(f"{len(traces)} traces; columns: {[c for c in ('trace_id','request','response') if c in traces.columns]}")

## 1. Human feedback on a trace

This is what a **👍/👎 button** in your app does under the hood: POST to a small endpoint that calls `log_feedback` on the request's trace. `source=HUMAN` records *who* judged.

In [ ]:
from mlflow.entities import AssessmentSource

tid = traces.iloc[0]["trace_id"]
mlflow.log_feedback(
    trace_id=tid,
    name="user_rating",
    value="thumbs_up",
    source=AssessmentSource(source_type="HUMAN", source_id="alice@example.com"),
    rationale="Accurate and concise.",
)
print("human feedback attached to", tid)

## 2. Automated (code) feedback on a trace

Not every signal needs a human or an LLM. A cheap deterministic check — length, a regex, a PII scan — can attach itself with `source=CODE`. Here we flag answers that are too long.

In [ ]:
tid2 = traces.iloc[1]["trace_id"]
answer_text = str(traces.iloc[1]["response"])
mlflow.log_feedback(
    trace_id=tid2,
    name="too_long",
    value=len(answer_text) > 400,
    source=AssessmentSource(source_type="CODE", source_id="length_check_v1"),
    rationale=f"answer length = {len(answer_text)} chars",
)
print("code feedback attached to", tid2)

## Read the assessments back

`get_trace(...).info.assessments` returns everything attached to a trace — and in the UI, each trace shows its feedback alongside the spans.

In [ ]:
for t in (tid, tid2):
    tr = mlflow.get_trace(t)
    print(t, "->", [(a.name, a.value, a.source.source_type) for a in (tr.info.assessments or [])])

## 3. Monitoring: score live traffic

Monitoring is just `c_`'s evaluation pointed at **production traces** instead of a test set: pull recent traces with `search_traces`, run a scorer over them, and watch the aggregate. A drop in `relevance_to_query/mean` is your regression alarm.

In [ ]:
from mlflow.genai.scorers import RelevanceToQuery

result = mlflow.genai.evaluate(data=traces, scorers=[RelevanceToQuery(model=JUDGE)])
print("quality over recent traffic:", result.metrics)

## OSS vs. managed — what you wire yourself

MLflow's **Review App** (a human-labeling UI), **labeling sessions**, and **scheduled scorers** (`ScorerScheduleConfig` — scorers that auto-run on every incoming production trace) are **Databricks-managed** features. In open-source MLflow you assemble the *same loop* from the primitives above:

- **Feedback collection** → your app's 👍/👎 control calls a small endpoint that runs `mlflow.log_feedback(...)`.
- **Automated monitoring** → a **cron job / scheduled script** runs the `search_traces` + `evaluate` from the monitoring cell over recent traffic and alerts when a metric drops.

The primitives are identical; only the scheduler and the labeling UI are managed.

## The GenAI track is complete

You've walked the GenAI MLOps spine, the parallel of the `ml/` track:

**trace (`a_`/`b_`/`d_`) → evaluate (`c_`) → version prompts (`e_`) → serve (`f_`) → feedback & monitor (`g_`).**

Optional, still on the board (see `roadmap/`):
- **DSPy** — `mlflow.dspy.autolog()` + automated prompt optimization.
- **LlamaIndex / Milvus** — a production-RAG notebook (real vector store) vs. a short appendix to `b_`.